# PushT demo notebook (Video 3)

Headings match the Parts of the Week 3 script. One kernel, top to bottom. If it dies, restart and run from the top of the current Part.

## Setup

In [ ]:
import json
import subprocess
import time
from pathlib import Path

import gymnasium as gym
import gym_pusht
import numpy as np
import torch

WEEK = Path("/Users/snudurupati/Projects/physical-ai-data-architect/weeks/week-03")
MIG  = str(WEEK / "model" / "diffusion_pusht_migrated")
SIM  = str(WEEK / "data" / "pusht_sim")
EVAL_DIR = WEEK / "outputs" / "dp_official_eval" / "videos" / "pusht_0"

## Part 3: the environment

PushT (`gym_pusht`, Pymunk 2-D physics, not MuJoCo): a pusher drags a T-block to a goal pose.

- Action = absolute target pixel in [0, 512]². The pusher chases it with a PD controller (k_p=100, k_v=20); `step()` = 3 physics substeps.
- Observation here = 5 numbers: pusher (x, y), block (x, y), block angle.
- Goal is fixed at [256, 256, π/4]. Success = coverage > 0.95 of the block's **center of mass**, which sits ~45 px above the geometric center.
- Gravity = 0, damping = 0: the block floats where it starts; only contact moves it.
- Episode cap: 300 steps. `info["coverage"]` and `info["is_success"]` are free ground-truth labels from the physics engine.

In [ ]:
env = gym.make("gym_pusht/PushT-v0", obs_type="state", render_mode="rgb_array")
obs, info = env.reset(seed=0)
print("obs (pusher_x, pusher_y, block_x, block_y, block_angle):")
print(obs)
print("goal_pose:", info["goal_pose"])

In [ ]:
print("action space low / high (absolute target pixels):")
print(env.action_space.low, env.action_space.high)

## Part 4: random policy — the baseline

`run_policy` rolls any `make_action(obs, info)` for `n_episodes` episodes and reports successes, mean failure length, and mean best coverage.

- Action: a fresh uniform sample in [0, 512]² every frame. No state read, no memory.
- Expected output: `0/100 successes, mean failure length 300, mean max coverage ~0.11` — coverage starts near 0.1 and nothing ever reaches 0.95.
- The point of the baseline: every later number is measured against it.

In [ ]:
def run_policy(make_action, n_episodes=100):
    successes, fail_lengths, max_covs = 0, [], []
    for ep in range(n_episodes):
        obs, info = env.reset(seed=ep)
        done, n, best = False, 0, 0.0
        while not done and n < 300:
            obs, r, term, trunc, info = env.step(make_action(obs, info))
            best = max(best, info["coverage"])
            done = term or trunc
            n += 1
        successes += int(term)
        if not term:
            fail_lengths.append(n)
        max_covs.append(best)
    print(f"{successes}/{n_episodes} successes, "
          f"mean failure length {np.mean(fail_lengths):.0f}, "
          f"mean max coverage {np.mean(max_covs):.3f}")

run_policy(lambda obs, info: env.action_space.sample().astype(np.float32))

## Part 5: the cowboy policy

The plan (drawn before coding it, `../assets/policy_v1.png`): stand on the far side of the block from the goal and shove it over. Code it as: aim the pusher at `block + 40 px` along the block→goal direction.

Why it fails — the physics has opinions:

- **margin 40:** the target lands on the block, so the pusher homes in, presses, and stops. Force through a PD controller with no momentum: it holds position, it does not sweep. Block never moves; coverage freezes at its spawn value (0.27).
- **margin 10:** the pusher now catches the T's thin arm. Force applied off the center of mass makes torque τ = r × F. With no gravity and no friction to pin the block, torque spins it instead of translating it. The T rotates in place forever.
- **What we assumed vs what is true:** we assumed pushing the middle moves the block to the goal; the block only translates when the force line passes through its center of mass, and coverage is measured *on the center of mass*, which is not the geometric center.
- Takeaway: a plausible diagram is not a control law. Contact geometry decides.

In [ ]:
def cowboy_policy(margin):
    def policy(obs, info):
        goal = np.array([info["goal_pose"][0], info["goal_pose"][1]])
        block = np.array([obs[2], obs[3]])
        d = block - goal
        norm = np.linalg.norm(d)
        u = d / norm if norm > 0 else np.array([0.0, 0.0])
        return (block + u * margin).astype(np.float32)
    return policy

run_policy(cowboy_policy(margin=40))

In [ ]:
run_policy(cowboy_policy(margin=10))   # the "cowboy": spins the T

One margin-40 episode, printed every 50 steps: the pusher arrives at its target and coverage never moves again.

In [ ]:
obs, info = env.reset(seed=0)
print("step | pusher (x, y) | action (x, y) | block (x, y) | coverage")
for step in range(300):
    action = cowboy_policy(margin=40)(obs, info)
    obs, r, term, trunc, info = env.step(action)
    if step % 50 == 0:
        print(f"{step:4d} | {obs[0]:6.1f} {obs[1]:6.1f} | "
              f"{action[0]:6.1f} {action[1]:6.1f} | "
              f"{obs[2]:6.1f} {obs[3]:6.1f} | {info['coverage']:.3f}")

## Part 6 (optional): the LLM-written policy

Asked Gemini to write a PushT policy. It produced clean code that scores 0/100. Two measured bugs:

- It hardcoded goal (400, 400). The real goal is (256, 256, π/4) — it aimed at an empty corner.
- It treated the action as a **displacement** from the pusher's position. The action is an **absolute target pixel**; adding small steps to it teleports the pusher around the table.
- Lesson: AI output still needs a reviewer who can read `env.action_space` and measure. The model cannot run your environment.

## Part 7: the checkpoint and the official eval

The published checkpoint (`lerobot/diffusion_pusht`, ~1 GB) is a DiffusionPolicy: it sees 2 image+state frames, runs a 16-step denoising loop, and emits 16 future actions, consuming 8 per step. About 38 model calls per episode.

- 0.4.4 loads weights through a *pre/post-processor* system (`policy_preprocessor.json` + normalization stats). This repo predates it, so `select_action` silently dropped normalization buffers — every rollout scored 0.
- The migration tool rebuilds the missing files; the eval then reproduces the README within noise: `pc_success 60.0` on 5 episodes (README: 65.4 on 500).

In [ ]:
!ls -lh {WEEK}/model/diffusion_pusht/ | tail -6

Migration command (run once; it produced `model/diffusion_pusht_migrated/`):

```
uv run python -m lerobot.processor.migrate_policy_normalization \
  --pretrained-path weeks/week-03/model/diffusion_pusht \
  --output-dir weeks/week-03/model/diffusion_pusht_migrated
```

Official eval command — 5 episodes, ~95 s each, so its console output (`pc_success: 60.0`) was captured earlier:

In [ ]:
!uv run python -m lerobot.scripts.lerobot_eval \
  --policy.path=weeks/week-03/model/diffusion_pusht_migrated \
  --env.type=pusht --eval.n_episodes=5 --eval.batch_size=1 \
  --policy.device=cpu --output_dir=weeks/week-03/outputs/dp_official_eval

A success clip from that eval (the T slides to the goal):

In [ ]:
subprocess.run(["open", str(EVAL_DIR / "eval_episode_1.mp4")])

## Part 8: the dataset writer

One `rollout(make_action, n_episodes, tag)` loop wrote both halves of the dataset. Its cells live in `sim_pushT.ipynb`; this cell shows only the frame schema.

- `add_frame` takes exactly the feature dict below; `is_success` comes from `info["is_success"]`, the physics engine's own label — free ground truth.
- The `task` tag ("random policy" / "diffusion") records **which policy** produced each frame, so the profiler can group by generator.
- Cost asymmetry, measured: `RANDOM 1000 done in 436s` vs `DIFFUSION 50 done in 5309s` (~106 s/episode). Cost per labeled sample is what sets how much ground truth you can afford.

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from lerobot.policies.diffusion.modeling_diffusion import DiffusionPolicy
from lerobot.policies.factory import make_pre_post_processors
from lerobot.envs.utils import preprocess_observation

features = {
    "observation.image": {"dtype": "image",   "shape": (96, 96, 3), "names": ["height", "width", "channel"]},
    "observation.state": {"dtype": "float32", "shape": (2,), "names": ["x", "y"]},
    "action":            {"dtype": "float32", "shape": (2,), "names": ["x", "y"]},
    "is_success":        {"dtype": "bool",    "shape": (1,), "names": None},
}
print(json.dumps(features, indent=1))

Dataset facts — the exact numbers the script quotes:

In [ ]:
info = json.loads((Path(SIM) / "meta" / "info.json").read_text())
print("total_episodes:", info["total_episodes"])
print("total_frames:  ", info["total_frames"])
print("fps:           ", info["fps"])
print("robot_type:    ", info["robot_type"])

# Per-task counts straight from the parquet files — no Spark session needed here.
import glob
import pyarrow.parquet as pq
from collections import defaultdict

task_names = {r["task_index"]: r["__index_level_0__"]
              for r in pq.read_table(Path(SIM) / "meta" / "tasks.parquet").to_pylist()}
eps, frames, successes = defaultdict(set), defaultdict(int), defaultdict(set)
for f in glob.glob(str(Path(SIM) / "data" / "*" / "*.parquet")):
    t = pq.read_table(f, columns=["task_index", "episode_index", "is_success"])
    for ti, ep, ok in zip(t.column("task_index").to_pylist(),
                          t.column("episode_index").to_pylist(),
                          t.column("is_success").to_pylist()):
        eps[ti].add(ep); frames[ti] += 1
        if ok: successes[ti].add(ep)
for ti in sorted(eps):
    print(f"{task_names[ti]:>14}: {len(eps[ti]):4d} episodes, {frames[ti]:6d} frames, "
          f"{len(successes[ti]):3d} success episodes")

Measured output of that cell:

```
random policy: 1000 episodes, 300000 frames,   0 success episodes
diffusion:       50 episodes,  10759 frames,  36 success episodes
```

Next for Parts 9–10: `01_profiler.ipynb` on the same dataset writes `results/pusht_sim.json`; its comparison cell puts it beside the two SO-101 profiles.